# Step 2. Register

*With IMPACT-Reg, from KonfAI: the only step that uses it.*

**Goal:** bring the moving image onto the fixed one, with one command, and get the transformation in OME-Zarr.

With a GPU, replace `--cpu 1` by `--gpu 0`. The first run takes longer: it downloads the preset and the elastix binary.

In [ ]:
!impact-reg-konfai register Generic_Rigid \
    -f ../output/data/fixed.ome.zarr -m ../output/data/moving.ome.zarr -o ../output/registration --cpu 1 --fields-only \
    --set Predictor.outputs_dataset.DisplacementField.OutputDataset.dataset_filename=Transform:omezarr

[ImpactReg] Registration completed in 11 s
  Preset:    Generic_Rigid (Generic Rigid)
  Fixed:     /path/to/ome-zarr-registration-tutorial/output/data/fixed.ome.zarr
  Moving:    /path/to/ome-zarr-registration-tutorial/output/data/moving.ome.zarr
  Transform: /path/to/ome-zarr-registration-tutorial/output/registration/P000/Transform.ome.zarr
  Record:    /path/to/ome-zarr-registration-tutorial/output/registration/register.json


The command prints its progress, then the summary kept above.

- **`-f` is the fixed image, `-m` the moving one.** Both are read from their stores, with their voxel sizes and units.
- **`Generic_Rigid` is a preset:** elastix, a rigid transformation, on any machine, in seconds.
- **`--fields-only` skips the resampled copy** of the moving image: this tutorial never writes one.
- **The last line sets the preset's output dataset.** `Transform:omezarr` writes the transformation in OME-Zarr. Without
  it, the preset writes `Transform.h5`, as ITK does.

You get two things:

In [ ]:
!ls ../output/registration ../output/registration/P000

../output/registration:
P000  register.json

../output/registration/P000:
Transform.ome.zarr


- `Transform.ome.zarr`: the transformation, an RFC-5 `affine`. A rigid registration is six parameters, so the preset
  declares `transform_type: affine` and writes the matrix it found: twelve numbers in `zarr.json`, no array. A deformable
  preset writes a `displacements` field instead, an image of vectors on the fixed grid. Steps 3 and 4 take either;
- `register.json`: what ran, with which versions.

## Which way does it go?

The transformation goes **from the fixed image to the moving one**, although it is the moving image that ends up
aligned. To draw the aligned image, a reader walks the grid of the fixed image and, for each point $x$ of it, asks where
that point is in the moving image:

$$I_{\mathrm{aligned}}(x) = I_{\mathrm{moving}}\big(T_{\mathrm{fixed} \rightarrow \mathrm{moving}}(x)\big)$$

So $T$ takes coordinates of the fixed image and gives coordinates of the moving one, in micrometers and in the order
z, y, x. A point of the fixed image goes through $T$ as it is; a point of the moving image would need its inverse.

## Try another preset

The engine is just a name. `impact-reg-konfai list` shows them all, and `impact-reg-konfai show <preset>` what each one
does and what it needs. Steps 3 and 4 run unchanged with any of them.

| Preset | Engine | Transformation | Time on this pair | Correlation | Nuclei mask Dice | Runs on |
|---|---|---|---|---|---|---|
| `Generic_Rigid` | elastix, rigid | `affine` | 15 s, CPU | 0.90 | 0.92 | any machine |
| `Generic_Rigid_BSpline` | elastix, rigid then B-spline | `displacements` | 4 min, CPU | 0.86 | 0.74 | any machine |
| `ConvexAdam_Composite` | ConvexAdam on MIND features | `displacements` | 1 min, GPU | 0.99 | 0.94 | Linux and Windows, no macOS package |
| `FireANTs_SyN` | FireANTs, diffeomorphic | `displacements` | 2 min 30, GPU | 0.99 | 0.94 | a GPU: over 50 min on a CPU |

Deformable is not always better: here, the B-spline scores below the rigid it starts from. The scores come from step 4;
correlation starts at -0.06, and the overlap at 0.

These presets declare `moments_init: com`: the moving image starts where the two subjects' centres of mass coincide. For
two images covering different lengths of a body, add `--set moments_init=cof`, which aligns the two frames instead.

**Next: [Step 3. Write, and read back](3-write.ipynb)**